# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamKapoor22/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

* **Finding 1 (Content Refresh Impact):** The paper states that refreshing stale content consistently recovers organic traffic within 30 days.
  * *Methodology Question:* Where does the baseline label come from, and does the evaluation control for broad seasonal market trends?
* **Finding 2 (Model Generalization):** The paper assumes the global model generalizes across all client domains equally.
  * *Methodology Question:* Does the validation design account for client-specific traffic patterns, or do high-volume clients skew metrics?

In [1]:
import os, sys
import pandas as pd

data_path = "data/raw/content_refresh_anonymized.csv"
raw_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path) if os.path.exists(data_path) else pd.read_csv(raw_url)
print("Data loaded successfully. Total rows:", len(df))

Data loaded successfully. Total rows: 30000


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

feature_cols = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'word_count']
X = df[feature_cols].fillna(0)
y = (df['trend_direction'].astype(str).str.lower() == 'down').astype(int)

# Before: Standard Random Split
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.2, random_state=42)
rf_rand = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr_r, y_tr_r)
f1_rand = f1_score(y_te_r, rf_rand.predict(X_te_r), zero_division=0)

# After: Honest Grouped Split
group_col = 'client_hash_id' if 'client_hash_id' in df.columns else df.columns[1]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, df[group_col]))
rf_honest = RandomForestClassifier(n_estimators=50, random_state=42).fit(X.iloc[tr_idx], y.iloc[tr_idx])
f1_honest = f1_score(y.iloc[te_idx], rf_honest.predict(X.iloc[te_idx]), zero_division=0)

print(f"Before (Random Split F1): {f1_rand:.4f}")
print(f"After (Honest Grouped Split F1): {f1_honest:.4f}")

Before (Random Split F1): 0.7098
After (Honest Grouped Split F1): 0.5648


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
print("--- Leakage Audit Check ---")
forbidden = ['future', 'target', 'label', 'trend']
leaked = [col for col in feature_cols if any(f in col.lower() for f in forbidden)]
if not leaked:
    print("PASS: No leakage columns found in features. Honest feature set verified.")
else:
    print("FAIL: Leaked columns:", leaked)

--- Leakage Audit Check ---
PASS: No leakage columns found in features. Honest feature set verified.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Safe Claim Rewrite:**
* *Old Claim:* "Our model guarantees traffic recovery for all pages."
* *Safe Rewritten Claim:* "Based on **observed** historical data and **measured** feature importances, the model serves as a **directional** indicator for **decision-support**."

In [4]:
print("Claim successfully rewritten using safe, public-facing terminology.")

Claim successfully rewritten using safe, public-facing terminology.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.